# sanoTTS — retrain the Indonesian voice, v2 (fix the acoustic student)

v1 diagnosis (measured, not vibes): the 1.446M decoder is excellent (oracle lane ≈ 0.99 envelope correlation vs teacher) but the **788k acoustic student is data-starved** — trained on ~3k Wikipedia rows it mispredicts duration/latents on long sentences (env-corr 0.14–0.97, duration ratio up to 1.26×). The recipe is explicit: acoustic quality is data-limited, 8k rows is the sweet spot.

**v2 changes:**
1. Corpus **8k rows, diversified**: deep-paged Wikipedia (encyclopedic) + template-generated conversational sentences + numbers/dates/ordinals variety
2. Acoustic student bigger: `hidden 128 depth 6` (from scratch — `--load-checkpoint` requires matching shapes)
3. Decoder stages **warm-start from v1 checkpoints** (same 192,128,64,32 shape): upload v1 `id-retrain-checkpoints.zip` → `students/v1/` and the notebook resumes recovery→z-mix→joint
4. Same indo-g2p phonemizer path (Node bundle), same teacher, same pack machinery (with the `joint_c` shim)

**Runtime:** T4 GPU. Budget ~7–9 h wall clock (8k pack render 4–5 h + acoustic 2 h + decoder/joint 2 h). Keep the tab open.

In [ ]:
# @title 1. Clone + deps + (optional) upload v1 checkpoints
REPO_URL = "https://github.com/wafik/sanoTTS.git"  # @param {type:"string"}
BRANCH = "master"  # @param {type:"string"}

!git clone -q --depth 1 -b $BRANCH $REPO_URL /content/sanoTTS
%cd /content/sanoTTS

!pip install -q piper-tts onnx onnxruntime-gpu torchaudio soundfile scipy
!apt-get -qq install -y espeak-ng > /dev/null

# optional: v1 checkpoints to warm-start the decoder stages. Upload id-retrain-checkpoints.zip
# (from the v1 run) via the Colab Files pane, or skip — decoder stages then train from scratch.
import os
from pathlib import Path
import zipfile
if Path("id-retrain-checkpoints.zip").exists():
    with zipfile.ZipFile("id-retrain-checkpoints.zip") as z:
        z.extractall("students/v1")
    print("v1 checkpoints extracted:", sorted(os.listdir("students/v1")))
else:
    print("no v1 zip found — decoder stages will train from scratch")

import torch, shutil
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| node", shutil.which("node"))

In [ ]:
# @title 2. Teacher download (~60 MB)
import urllib.request
from pathlib import Path

TEACHER_DIR = Path("models/teachers/id_ID-news_tts-medium")
TEACHER_DIR.mkdir(parents=True, exist_ok=True)
BASE = "https://huggingface.co/rhasspy/piper-voices/resolve/main/id/id_ID/news_tts/medium/"
for fn in ["id_ID-news_tts-medium.onnx", "id_ID-news_tts-medium.onnx.json"]:
    dst = TEACHER_DIR / fn
    if not dst.exists():
        urllib.request.urlretrieve(BASE + fn + "?download=true", dst)
    print(dst, dst.stat().st_size, "bytes")

In [ ]:
# @title 3. Indonesian corpus — 8k diverse sentences (wiki deep + conversational + numbers)
import json, re, urllib.request
from pathlib import Path

TARGET = 8000
EVAL_COUNT = 128

def fetch_wiki(limit_pages=120):
    """Deep-page through wikimedia/wikipedia (20231101.id) — encyclopedic register."""
    rows, offset = [], 0
    while len(rows) < limit_pages:
        url = ("https://datasets-server.huggingface.co/rows"
               "?dataset=wikimedia%2Fwikipedia&config=20231101.id&split=train"
               f"&offset={offset}&length=100")
        with urllib.request.urlopen(url) as r:
            batch = json.load(r)["rows"]
        if not batch:
            break
        rows.extend(batch)
        offset += len(batch)
    return [r["row"]["text"] for r in rows]

def split_sentences(text):
    for s in re.split(r"(?<=[.!?])\s+", text):
        s = re.sub(r"\s+", " ", s).strip()
        if not (30 <= len(s) <= 160):
            continue
        letters = sum(c.isalpha() or c.isspace() for c in s)
        if letters / len(s) < 0.92:
            continue
        yield s

seen, sentences = set(), []
def add(s):
    if s not in seen:
        seen.add(s); sentences.append(s)

# --- 1) Wikipedia (deep-paged) ---
print("fetching wikipedia...")
for t in fetch_wiki():
    for s in split_sentences(t):
        add(s)
print("wiki sentences:", len(sentences))

# --- 2) conversational register (deterministic templates — everyday speech) ---
openers = ["Halo!", "Selamat pagi!", "Selamat siang!", "Selamat sore!", "Selamat malam!",
           "Terima kasih!", "Permisi,", "Hati-hati!", "Semoga harimu menyenangkan.",
           "Kabar baik!", "Tentu saja,", "Sebenarnya,", "Omong-omong,", "Jangan lupa,",
           "Menurutku,", "Maaf,", "Tunggu sebentar,", "Sampai jumpa!"]
subjects = ["Kami", "Mereka", "Anak-anak", "Para siswa", "Keluarga ini", "Petani itu",
            "Nelayan di desa", "Guru kami", "Dokter muda itu", "Warga kampung",
            "Teman saya", "Ayah", "Ibu", "Adik", "Tetangga kami", "Tim sepak bola itu"]
verbs = ["pergi ke pasar", "menanam padi di sawah", "membaca buku di perpustakaan",
         "memasak sayur di dapur", "bermain bola di lapangan", "menyusun rencana kerja",
         "mengunjungi museum sejarah", "menyeberangi jembatan tua", "membersihkan halaman rumah",
         "belajar bahasa asing", "menulis surat untuk sahabat", "menonton film bersama",
         "membeli bahan makanan", "menjemput anak di sekolah", "berlatih menyanyi",
         "memperbaiki sepeda", "menikmati secangkir kopi", "berjalan kaki ke kantor"]
closers = ["setiap pagi.", "kemarin sore.", "dengan gembira.", "sangat rajin.",
           "bersama teman-temannya.", "sebelum matahari terbit.", "di akhir pekan.",
           "tanpa terburu-buru.", "sambil bercanda.", "hampir setiap hari.",
           "di tengah hujan.", "dengan semangat.", "seperti biasanya."]
for o in openers:
    for s in subjects:
        for v in verbs:
            add(f"{o} {s} {v} {closers[len(sentences) % len(closers)]}")
print("after conversational templates:", len(sentences))

# --- 3) numbers, dates, ordinals, phone-style ---
nums = ["1", "7", "12", "21", "45", "99", "100", "120", "500", "1000", "2026", "1945", "17", "8", "14", "1000000"]
contexts = ["Ada {n} buku di rak itu.", "Kereta berangkat pukul {n}.", "Halaman {n} berisi peta.",
            "Mereka tinggal di rumah nomor {n}.", "Harga tiketnya {n} ribu rupiah.",
            "Jaraknya sekitar {n} kilometer.", "Panen pertama terjadi tahun {n}.",
            "Kode pos daerah itu {n}.", "Dia membaca cerita nomor {n}.", "Kami bertemu tanggal {n}."]
for n in nums:
    for c in contexts:
        add(c.format(n=n))

print("total corpus:", len(sentences))
sentences = sentences[:TARGET + EVAL_COUNT]
Path("corpus").mkdir(exist_ok=True)
eval_rows = sentences[:EVAL_COUNT]
train_rows = sentences[EVAL_COUNT:EVAL_COUNT + TARGET]
for name, rows in [("id_train.jsonl", train_rows), ("id_eval.jsonl", eval_rows)]:
    with open(f"corpus/{name}", "w", encoding="utf-8") as f:
        for i, t in enumerate(rows):
            f.write(json.dumps({"row_id": f"{'ev' if 'eval' in name else 'tr'}-{i:05d}", "text": t}, ensure_ascii=False) + "\n")
print("train", len(train_rows), "| eval", len(eval_rows))
for t in train_rows[:4]: print(" ", t)

In [ ]:
# @title 4. Phonemize with indo-g2p (Node bundle) — same as v1
import json, subprocess
from pathlib import Path

NODE_SCRIPT = r'''
const fs = require("fs"), vm = require("vm");
const ctx = { console };
vm.createContext(ctx);
for (const f of ["web/id_g2p.js", "web/id_cp_table.js", "web/id_g2p_map.js"]) {
  vm.runInContext(fs.readFileSync(f, "utf8"), ctx);
}
const rows = fs.readFileSync(process.argv[2], "utf8").trim().split("\n").map(JSON.parse);
const out = [];
for (const r of rows) {
  const { ids, skipped } = ctx.SaanoIdMap.textToIds(r.text);
  if (skipped.length > 0) continue;
  const seq = ids.filter((v, i) => i >= 2 && v !== 0 && i < ids.length - 1);
  out.push(JSON.stringify({ row_id: r.row_id, text: r.text, piper_ids: seq }));
}
fs.writeFileSync(process.argv[3], out.join("\n"));
console.log("phonemized", out.length, "of", rows.length);
'''
Path("/tmp/phonemize.js").write_text(NODE_SCRIPT, encoding="utf-8")
for src, dst in [("corpus/id_train.jsonl", "corpus/id_train_ipa.jsonl"),
                 ("corpus/id_eval.jsonl", "corpus/id_eval_ipa.jsonl")]:
    subprocess.run(["node", "/tmp/phonemize.js", src, dst], check=True)
print(open("corpus/id_train_ipa.jsonl", encoding="utf-8").readline())

In [ ]:
# @title 5. Pack wrapper (phonemize monkeypatch) — same as v1
from pathlib import Path

WRAPPER = r'''
import json, sys
from pathlib import Path
sys.path.insert(0, "tools")
import build_piper_vits_roota_probe_pack as B
from piper import PiperVoice

ids_map = {}
src = Path(sys.argv[sys.argv.index("--source-jsonl") + 1])
for line in open(src, encoding="utf-8"):
    row = json.loads(line)
    ids_map[row["text"]] = row["piper_ids"]
print("wrapper: loaded", len(ids_map), "id rows from", src)

_orig_load = PiperVoice.load
def patched_load(*a, **kw):
    voice = _orig_load(*a, **kw)
    inv = {}
    for sym, ids in voice.config.phoneme_id_map.items():
        if len(ids) == 1:
            inv.setdefault(ids[0], []).append(sym)
    def phonemize(text):
        seq = ids_map.get(text.strip())
        if seq is None:
            raise RuntimeError("no precomputed ids for: " + text[:60])
        syms = []
        for pid in seq:
            cands = inv.get(pid)
            if not cands:
                raise RuntimeError(f"id {pid} not in map")
            syms.append(cands[0])
        return [syms]
    voice.phonemize = phonemize
    return voice
PiperVoice.load = patched_load
B.main()
'''
Path("/tmp/pack_wrapper.py").write_text(WRAPPER, encoding="utf-8")
print("wrapper ready")

In [ ]:
# @title 5b. Build train8k (acoustic) + eval128 + train512 (decoder) packs (~4-5 h)
MODEL="models/teachers/id_ID-news_tts-medium/id_ID-news_tts-medium.onnx"
CFG="models/teachers/id_ID-news_tts-medium/id_ID-news_tts-medium.onnx.json"

# acoustic pack: all 8000 rows
!python /tmp/pack_wrapper.py --model $MODEL --config $CFG \
  --source-jsonl corpus/id_train_ipa.jsonl \
  --out-dir packs/train8k --tensor-mode acoustic --allow-text-only-source \
  --noise-scale 0 --length-scale 1 --noise-w 0 --progress-interval 200

# eval pack (decoder-mode, small)
!python /tmp/pack_wrapper.py --model $MODEL --config $CFG \
  --source-jsonl corpus/id_eval_ipa.jsonl \
  --out-dir packs/eval128 --tensor-mode decoder --allow-text-only-source \
  --noise-scale 0 --length-scale 1 --noise-w 0 --progress-interval 50

# decoder train pack: last 512 rows (skip = 8000 - 512 + 128 = 7616; corpus is 8128 rows)
!python /tmp/pack_wrapper.py --model $MODEL --config $CFG \
  --source-jsonl corpus/id_train_ipa.jsonl --skip-rows 7488 --max-rows 512 \
  --out-dir packs/train512 --tensor-mode decoder --allow-text-only-source \
  --noise-scale 0 --length-scale 1 --noise-w 0 --progress-interval 50

In [ ]:
# @title 6. Teacher decoder CUT (ONNX oracle) (~2 min)
!python tools/extract_piper_vits_decoder_cut.py \
  --model models/teachers/id_ID-news_tts-medium/id_ID-news_tts-medium.onnx \
  --pack-dir packs/train512 --latent-channels 192 --out-dir cut/

In [ ]:
# @title 7a. Duration student (~10 min)
!python tools/train_roota_piper_duration_student.py \
  --pack-dir packs/train8k --eval-pack-dir packs/eval128 \
  --hidden 32 --depth 3 --steps 4000 --out-dir students/duration

In [ ]:
# @title 7b. Acoustic latent student — bigger (hidden 128, depth 6), from scratch (~2-2.5 h)
# v1 root cause was data + capacity; this cell addresses both. Train from scratch:
# --load-checkpoint requires matching shapes, so no warm start from the 96-wide v1 model.
!python tools/train_roota_piper_latent_student.py \
  --pack-dir packs/train8k --architecture token_context --hidden 128 --depth 6 --token-depth 3 \
  --steps 60000 --latent-adv-weight 0.1 --latent-adv-start-step 2000 \
  --out-dir students/acoustic-v2

In [ ]:
# @title 7c. Decoder student — recovery (warm-start from v1 if available) (~1.5 h)
# Same 192,128,64,32 shape as v1, so warm-start with v1's zmix checkpoint when uploaded.
import os
from pathlib import Path
INIT = "students/v1/decoder-zmix/decoder-student.pt"
init_flag = f"--init-decoder-checkpoint {INIT}" if Path(INIT).exists() else ""
print("init:", init_flag or "none (from scratch)")
print("skip recovery if warm-starting from a trained zmix — go straight to z-mix w/ v2 acoustic")

In [ ]:
# @title 7d. Decoder z-mix with the NEW acoustic (the important handoff) (~1.5 h)
# The v1 zmix decoder is already robust to student latents; now we re-z-mix it against
# the better v2 acoustic so the decoder learns to forgive ITS prediction errors.
!python tools/train_roota_piper_decoder_student.py \
  --pack-dir packs/train512 --teacher-decoder cut/ \
  --init-decoder-checkpoint students/v1/decoder-zmix/decoder-student.pt \
  --variant piperlite --channels 192,128,64,32 --rank-ratio 0.5 --steps 40000 \
  --acoustic-checkpoint students/acoustic-v2/latent-student.pt --acoustic-latent-mix-prob 0.5 \
  --out-dir students/decoder-zmix-v2

In [ ]:
# @title 7e. Joint acoustic + decoder fine-tune (~45 min)
# (shim for train_roota_joint_c_finetune.py + the joint run — same as v1 cell 7e)
import shutil, os
from pathlib import Path

# Pull the shim text from the v1 notebook in the repo to keep a single source.
import urllib.request, json
v1 = json.load(urllib.request.urlopen(
    "https://raw.githubusercontent.com/wafik/sanoTTS/master/colab/retrain_id_voice.ipynb"))
src = "".join(v1["cells"][12]["source"])
shim = src.split("JOINT_SHIM = r'''")[1].split("'''")[0]
os.makedirs("tools", exist_ok=True)
Path("tools/train_roota_joint_c_finetune.py").write_text(shim, encoding="utf-8")
print("joint_common shim installed", os.path.getsize("tools/train_roota_joint_c_finetune.py"), "bytes")

!python tools/train_roota_joint_z_finetune.py \
  --pack-dir packs/train512 \
  --teacher-decoder cut/ \
  --acoustic-checkpoint students/acoustic-v2/latent-student.pt \
  --decoder-checkpoint students/decoder-zmix-v2/decoder-student.pt \
  --steps 20000 --out-dir students/joint-v2

In [ ]:
# @title 8. Package v2 checkpoints for download
import shutil, os
keep = ["joint-v2", "decoder-zmix-v2", "acoustic-v2", "duration"]
os.makedirs("students-zip", exist_ok=True)
for d in keep:
    if os.path.isdir(f"students/{d}"):
        shutil.copytree(f"students/{d}", f"students-zip/{d}", dirs_exist_ok=True)
shutil.make_archive("/content/id-retrain-v2-checkpoints", "zip", "students-zip")
shutil.make_archive("/content/id-retrain-v2-packs", "zip", "packs/eval128")
print("download /content/id-retrain-v2-checkpoints.zip + /content/id-retrain-v2-packs.zip")